In [1]:
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn
from google import genai
import PyPDF2
from time import sleep

NLTK_DATA_DIR='.venv//share//nltk_data'


## Step 1: note down emotional words

Disclaimer: the following lists were generated using VS Code Copilot assist and verified and modified manually. 

In [2]:
from abc import ABC, abstractmethod

In [6]:
class EmoWordsBase(ABC):
    def __init__(self):
        self.happy_words = ['happy','joyful','content','pleased','delighted','cheerful','merry','jovial','ecstatic','elated']
        self.struggling_words = ['struggle','difficulty','challenge','hardship','obstacle','adversity','trial','tribulation','ordeal','setback']
        self.surprised_words = ['surprise','astonishment','amazement','wonder','shock','disbelief','stunned','startled','flabbergasted']
        self.annoyed_words = ['annoyed','irritated','frustrated','displeased','vexed','aggravated','exasperated','discontented']
        self.unhappy_words = ['unhappy','sad','depressed','miserable','downcast','gloomy','melancholy','sorrowful','heartbroken','despondent']
        self.interested_words = ['interested','curious','fascinated','engaged','intrigued','captivated','enthusiastic', 'eager']
        self.appreciative_words = ['appreciative','grateful','thankful','admired','respected','valued']
        self.nervous_words = ['nervous','anxious','uneasy','apprehensive','worried','tense','restless']
        
        self.dict_of_emotional_words = {'happiness': self.happy_words,
                                'struggle': self.struggling_words,
                                'surprise': self.surprised_words,
                                'annoyance': self.annoyed_words,
                                'unhappiness': self.unhappy_words,
                                'interest': self.interested_words,
                                'appreciation': self.appreciative_words,
                                'nervousness': self.nervous_words}
        
    
    def get_emotional_words(self, emotion):
        return self.dict_of_emotional_words.get(emotion, [])
    
    def add_emotional_words(self, emotion, words):
        if emotion in self.dict_of_emotional_words:
            self.dict_of_emotional_words[emotion].extend(words)
        else:
            self.dict_of_emotional_words[emotion] = words
    
    def remove_emotional_word(self, emotion, word):
        if emotion in self.dict_of_emotional_words:
            self.dict_of_emotional_words[emotion] = [w for w in self.dict_of_emotional_words[emotion] if w != word]
    
    def stem_emotional_words(self, stemmer = nltk.stem.SnowballStemmer('english')):
        self.stemmer = stemmer
        self.dict_of_emotional_stems = {}
        for key in self.dict_of_emotional_words.keys():
            self.dict_of_emotional_stems[key] = [self.stemmer.stem(word) for word in self.dict_of_emotional_words[key]]
        return self.dict_of_emotional_stems
    
    def lemmatize_emotional_words(self, lemmatizer = nltk.stem.WordNetLemmatizer()):
        self.lemmatizer = lemmatizer
        self.dict_of_emotional_lemmas = {}
        for key in self.dict_of_emotional_words.keys():
            self.dict_of_emotional_lemmas[key] = [self.lemmatizer.lemmatize(word) for word in self.dict_of_emotional_words[key]]
        return self.dict_of_emotional_lemmas


class EmoWordsFinder(EmoWordsBase):
    def __init__(self,sentences=None):
        super().__init__()
        if sentences is None:
            self.sentences = self.generate_test_portfolio()
        else:
            self.sentences = sentences
    
    def find_emotional_words_by_stem(self, method = 'AI'):
        if self.stemmer is None:
            self.stemmer=nltk.stem.SnowballStemmer('english')
        if method=='AI':
            self.client = genai.Client()
            self.sentences_found_ai={}
            self.words_found_ai = {}
            for key in self.dict_of_emotional_stems.keys():
                self.sentences_found_ai[key] = []
                self.words_found_ai[key] = []
                
            for sentence in self.sentences:
                sentence=sentence.replace('\n', '')
                for word in sentence.split():
                    for key in self.dict_of_emotional_stems.keys():
                        
                        if self.stemmer.stem(word.strip()) in self.dict_of_emotional_stems[key]:
                            try:
                                response = self.client.models.generate_content(
                                    model="gemini-3.1-flash-lite", 
                                    contents=f"If the following sentence uses the word {word} to express emotions of the {key} type, then write the word emotional, else write the word nonemotional. Sentence: {sentence}"
                                )
                            except:
                                sleep(60)
                            if response.text == 'emotional':
                                self.sentences_found_ai[key].append(sentence)
                                self.words_found_ai[key].append(word)
            self.words_found=self.words_found_ai
            self.sentences_found=self.sentences_found_ai
        
        elif method=='manual':
            self.sentences_found_manual={}
            self.words_found_manual={}
            for key in self.dict_of_emotional_stems.keys():
                self.sentences_found_manual[key] = []
                self.words_found_manual[key] = []
            for sentence in self.sentences:
                sentence=sentence.replace('\n', '')
                i=0
                while i<len(sentence.split()):
                    word = sentence.split()[i]
                    for key in self.dict_of_emotional_stems.keys():
                        
                        if self.stemmer.stem(word.strip()) in self.dict_of_emotional_stems[key]:
                            print(f"Emotional word found:/n emotion: {key}, word: {word}/n in: {sentence}")
                            user_input = input("Is this an emotional word? (y/n): ")
                            
                            if user_input.lower() == 'y':
                                self.sentences_found_manual[key].append(sentence)
                                self.words_found_manual[key].append(word)
                                i+=1
                            elif user_input.lower() == 'n':
                                i+=1
                            else:
                                print("Invalid input. Please enter 'y' or 'n'.")

    def find_emotional_words_by_lemma(self, method = 'AI'):
        
        if self.lemmatizer is None:
            self.lemmatizer = nltk.stem.WordNetLemmatizer()
            
        if method=='AI':
            self.client = genai.Client()
            self.sentences_found_ai={}
            self.words_found_ai = {}
            for key in self.dict_of_emotional_stems.keys():
                self.sentences_found_ai[key] = []
                self.words_found_ai[key] = []
                
            for sentence in self.sentences:
                sentence=sentence.replace('\n', '')
                for word in sentence.split():
                    for key in self.dict_of_emotional_stems.keys():
                        
                        if self.lemmatizer.lemmatize(word.strip()) in self.dict_of_emotional_stems[key]:
                            try:
                                response = self.client.models.generate_content(
                                    model="gemini-3.1-flash-lite", 
                                    contents=f"If the following sentence uses the word {word} to express emotions of the {key} type, then write the word emotional, else write the word nonemotional. Sentence: {sentence}"
                                )
                            except:
                                sleep(60)
                            if response.text == 'emotional':
                                self.sentences_found_ai[key].append(sentence)
                                self.words_found_ai[key].append(word)
            self.words_found=self.words_found_ai
            self.sentences_found=self.sentences_found_ai
        
        elif method=='manual':
            self.sentences_found_manual={}
            self.words_found_manual={}
            for key in self.dict_of_emotional_stems.keys():
                self.sentences_found_manual[key] = []
                self.words_found_manual[key] = []
            for sentence in self.sentences:
                sentence=sentence.replace('\n', '')
                i=0
                while i<len(sentence.split()):
                    word = sentence.split()[i]
                    for key in self.dict_of_emotional_stems.keys():
                        
                        if self.lemmatizer.lemmatize(word.strip()) in self.dict_of_emotional_stems[key]:
                            print(f"Emotional word found:/n emotion: {key}, word: {word}/n in: {sentence}")
                            user_input = input("Is this an emotional word? (y/n): ")
                            
                            if user_input.lower() == 'y':
                                self.sentences_found_manual[key].append(sentence)
                                self.words_found_manual[key].append(word)
                                i+=1
                            elif user_input.lower() == 'n':
                                i+=1
                            else:
                                print("Invalid input. Please enter 'y' or 'n'.")
            self.words_found=self.words_found_manual
            self.sentences_found=self.sentences_found_manual
                                
                                
    def generate_test_portfolio(self, n_sentences=1):
        self.client = genai.Client()
        self.test_portfolio = []
        for emotion in self.dict_of_emotional_words.keys():
            for _ in range(n_sentences):
                response = self.client.models.generate_content(
                    model="gemini-3.1-flash-lite",
                    contents=f"Generate one sentence that expresses the emotion of {emotion} using one of the following words: {self.dict_of_emotional_words[emotion]} in the context of a portfolio for a course on data visualisations. The word can be used in any form as long as it expresses the emotion of the student in the context of the course. For example for the emotion of excitement: I was excited about being able to explore visualisations for this dataset in a more relaxed environment. Respond with only the sentence, without any additional text."
                )
                
                self.test_portfolio.append(response.text)
            
            response = self.client.models.generate_content(
                    model="gemini-3.1-flash-lite",
                    contents=f"Generate one sentence that uses one of the following words: {self.dict_of_emotional_words[emotion]} in the context of a portfolio for a course on data visualisations but does not use the word to express an emotion. For example: The values on the X-axis represent the hormone levels. Respond with only the sentence, without any additional text. If you cannot generate such a sentence, respond with the word 'impossible'."
                )
            
            if response.text!='impossible':
                self.test_portfolio.append(response.text)
        return self.test_portfolio

                


        
        

In [7]:
emofinder =  EmoWordsFinder()

In [8]:
emofinder.sentences

['I am delighted by how much I have grown in my ability to translate complex data into clear, compelling visual narratives throughout this course.',
 "The color palette designated as 'cheerful' identifies the cluster of outliers within the scatter plot.",
 'Navigating the intricacies of complex data visualization tools proved to be a significant challenge that tested my patience and deepened my resolve to master the craft.',
 'The primary challenge in designing this dashboard was mapping the multi-layered temporal data onto a two-dimensional scatter plot.',
 'I was absolutely flabbergasted by how much clarity a well-structured visualization could bring to such a complex and messy dataset.',
 'The data points clustered within the scatter plot reveal a genuine surprise in the distribution of the sample population.',
 'I felt increasingly frustrated as the complex software failed to render my data visualisations exactly how I had envisioned them for the final portfolio.',
 'The frustrated

In [ ]:
happy_words = ['happy','joyful','content','pleased','delighted','cheerful','merry','jovial','ecstatic','elated']
struggle_words = ['struggle','difficulty','challenge','hardship','obstacle','adversity','trial','tribulation','ordeal','setback']
surprisal_words = ['surprise','astonishment','amazement','wonder','shock','disbelief','awe','stunned','startled','flabbergasted']
annoyance_words = ['annoyance','irritation','frustration','displeasure','vexation','aggravation','exasperation','discontent','resentment','grumpiness']
unhappy_words=['unhappy','sad','depressed','miserable','downcast','gloomy','melancholy','sorrowful','heartbroken','despondent']
interest_words = ['interest','curiosity','fascination','engagement','intrigue','captivation','absorption','enthusiasm','eagerness','zeal']
appreciation_words = ['appreciation','gratitude','thankfulness','recognition','acknowledgment','admiration','respect','esteem','regard','value']
nervous_words=['nervous','anxious','uneasy','apprehensive','worried','tense','edgy','jittery','restless','fidgety']

In [ ]:
dict_of_emotional_words = {'happy': happy_words,
        'struggling': struggle_words,
        'surprised': surprisal_words,
        'annoyed': annoyance_words,
        'unhappy': unhappy_words,
        'interested': interest_words,
        'appreciative': appreciation_words,
        'nervous': nervous_words}

dict_of_emotional_stems = {}

In [ ]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("english")

In [ ]:
for key in dict_of_emotional_words.keys():
    dict_of_emotional_stems[key] = [stemmer.stem(word) for word in dict_of_emotional_words[key]]

In [ ]:
import PyPDF2


In [ ]:
full_text=''
with open("Portfolios/sampleportfolio.pdf", "rb") as pdf_file:
    read_pdf = PyPDF2.PdfReader(pdf_file)
    for page in read_pdf.pages:
        full_text += page.extract_text()
sentences = full_text.split('.')

In [ ]:
from google import genai


client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.1-flash-lite", contents="Explain how AI works in a few words"
)
print(response.text)

In [ ]:
from time import sleep

In [ ]:
sentences_found={}
for sentence in sentences:
    sentence=sentence.replace('\n', '')
    for word in sentence.split():
        for key in dict_of_emotional_stems.keys():
            if stemmer.stem(word.strip()) in dict_of_emotional_stems[key]:
                print(f"{key}, {word} in: {sentence}")
                try:
                    response = client.models.generate_content(
                        model="gemini-3.1-flash-lite", 
                        contents=f"If the following sentence uses the word {word} to express emotions of the {key} type, then write the word emotional, else write the word nonemotional. Sentence: {sentence}"
                    )
                except:
                    sleep(60)
                print(response.text)
                sentences_found[key].append(sentence.replace('\n', ''))
                break


In [ ]:
print(sentences_found)